In [ ]:
pip install torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 15.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 20.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 58.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 19.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 49.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 55.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 46.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 39.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Bloco 1 - Imports, download (opcional) e seed

import os
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# (Opcional) baixar e descompactar o dataset dentro do notebook:
# Descomente se ainda não tiver o zip baixado.
# !wget https://github.com/SVizor42/ML_Zoomcamp/releases/download/straight-curly-data/data.zip
# !unzip -q data.zip

# Reprodutibilidade
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


ModuleNotFoundError: No module named 'torchvision'

In [ ]:
# Bloco 2 - Datasets e DataLoaders (sem augmentations)

data_dir = "data"  # pasta criada ao descompactar data.zip
train_dir = os.path.join(data_dir, "train")
test_dir = os.path.join(data_dir, "test")  # usaremos como "validation/test"

# Transforms para treino e validação (sem augmentations por enquanto)
base_transforms = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder(train_dir, transform=base_transforms)
validation_dataset = datasets.ImageFolder(test_dir, transform=base_transforms)

batch_size = 20

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)

print("Tamanho do train_dataset:", len(train_dataset))
print("Tamanho do validation_dataset:", len(validation_dataset))
print("Classes:", train_dataset.classes)


In [ ]:
# Bloco 3 - Definição do modelo, loss (Q1) e contagem de parâmetros (Q2)

class HairNet(nn.Module):
    def __init__(self):
        super(HairNet, self).__init__()
        # Input: (3, 200, 200)
        self.conv = nn.Conv2d(
            in_channels=3,
            out_channels=32,
            kernel_size=(3, 3),  # sem padding -> 200 -> 198
            padding=0
        )
        self.pool = nn.MaxPool2d(kernel_size=(2, 2))  # 198 -> 99
        # Após conv+pool: (32, 99, 99)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 99 * 99, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 1)  # saída escalar para classificação binária

    def forward(self, x):
        x = self.conv(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)  # logits (NÃO aplicamos sigmoid aqui)
        return x

model = HairNet().to(device)

# Q1: função de perda adequada para saída de 1 neurônio com logits
criterion = nn.BCEWithLogitsLoss()

# Otimizador especificado
optimizer = torch.optim.SGD(model.parameters(), lr=0.002, momentum=0.8)

# Q2: contagem de parâmetros
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")


In [ ]:
# Bloco 4 - Treino inicial (10 épocas) + estatísticas Q3 e Q4

num_epochs = 10
history = {'acc': [], 'loss': [], 'val_acc': [], 'val_loss': []}

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        # labels como float e shape (batch_size, 1)
        labels = labels.float().unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(images)                 # logits
        loss = criterion(outputs, labels)      # BCEWithLogitsLoss espera logits
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        # Para acurácia: aplica sigmoid e threshold 0.5
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct_train / total_train
    history['loss'].append(epoch_loss)
    history['acc'].append(epoch_acc)

    # Validação
    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)

            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_epoch_loss = val_running_loss / len(validation_dataset)
    val_epoch_acc = correct_val / total_val
    history['val_loss'].append(val_epoch_loss)
    history['val_acc'].append(val_epoch_acc)

    print(f"Epoch {epoch+1}/{num_epochs}, "
          f"Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}, "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}")

# ---- Estatísticas para Q3 e Q4 ----
acc_array = np.array(history['acc'])
loss_array = np.array(history['loss'])

median_train_acc = np.median(acc_array)
std_train_loss = np.std(loss_array)

print("\n=== Estatísticas das 10 épocas (sem augmentation) ===")
print(f"Mediana da acurácia de treino (Q3): {median_train_acc:.4f}")
print(f"Desvio padrão da loss de treino (Q4): {std_train_loss:.4f}")
